# Sprint 5: Late-fusion policy engine + NRRP validation

Proposal weeks 9–10: concatenate the Attention U-Net flood probability and the GraphSAGE deprivation probability vector, look up a **rule-based three-class policy matrix**, write a parcel GeoJSON layer for Kibera / Mathare / Mukuru, and check that **Compensated Resettlement** parcels concentrate inside the NRRP 37 high-risk zones (or an OSM river-corridor fallback).

| Output | Meaning |
|--------|---------|
| Retain and Formalise | Safe ground, low deprivation — tenure-security candidate |
| Retain and Upgrade | In-situ drainage / WASH / housing investment |
| Compensated Resettlement | Acute flood (NRRP-aligned relocate class) |

**What this notebook does**

1. Checks out the Sprint 5 branch (or `main` once merged).
2. Loads (or rebuilds) the three-settlement graphs **and** building-footprint GeoJSON.
3. Runs GraphSAGE `predict_proba` from the Sprint 4 checkpoint when present.
4. Builds a flood-probability surface (trained U-Net when a checkpoint + raster stack exist; otherwise samples the event flood-label band and says so).
5. Fuses the two heads with `FusionEngine` and writes `outputs/classifications/{settlement}_policy.geojson`.
6. Scores parcel-level **resettlement lift** against NRRP polygons or the river-corridor fallback.

XAI overlays and the Folium dashboard are **Sprint 6** — this notebook stops at the classified parcel layer.

**Offline / Cloud Agent note.** Real checkpoints, GEE stacks, NRRP polygons, and Overpass pulls only run here in Colab. `src/models/fusion_engine.py` and `src/utils/parcel_io.py` are covered by offline unit tests with synthetic parcels.


## 0. Before you run

- **Runtime → Change runtime type → T4 GPU** recommended if you will re-run U-Net sliding-window inference; GraphSAGE fusion is fine on CPU.
- Sprint 4 checkpoints (`graphsage_best.pt`, `attention_unet_best.pt`) and tiled stacks should already live on Drive from notebooks 02–04.
- Optional: set `NRRP_GEOJSON_PATH` to the 37 high-risk zone polygons. If it is `None`, the notebook buffers OSM rivers/streams — the same fallback Sprint 2 v2 uses for flood-label lift.
- Mount Drive so graphs, rasters, checkpoints, and the written GeoJSON persist across Colab disconnects.


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU — fusion is CPU-cheap; U-Net sliding-window inference will be slower.")

In [ ]:
%%bash
# Prefer the Sprint 5 PR branch while open; switch BRANCH to main after merge.
BRANCH=cursor/sprint5-fusion-engine-b5c4
REPO=https://github.com/emmanuelamor5/nairobi-risk-mapping.git

if [ -d /content/nairobi-risk-mapping/.git ]; then
  cd /content/nairobi-risk-mapping
  git fetch origin "$BRANCH" || git fetch origin main
  git checkout "$BRANCH" 2>/dev/null || git checkout main
  git pull --ff-only origin "$BRANCH" 2>/dev/null || git pull --ff-only origin main
else
  git clone "$REPO" /content/nairobi-risk-mapping
  cd /content/nairobi-risk-mapping
  git fetch origin "$BRANCH" || true
  git checkout "$BRANCH" 2>/dev/null || git checkout main
fi
cd /content/nairobi-risk-mapping
git rev-parse --abbrev-ref HEAD
ls src/models/fusion_engine.py src/utils/parcel_io.py

In [ ]:
%cd /content/nairobi-risk-mapping
!pip install -q -r requirements.txt

In [ ]:
# --- config knobs ---
from pathlib import Path
import os
import sys

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/nairobi_risk_mapping")
except ImportError:
    print("Not in Colab — artefacts stay under the repo working directory.")
    DRIVE_ROOT = Path(".")

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

# Symlink Drive data/ into the repo working tree when running in Colab so
# DataPipeline's relative paths (data/graphs, data/tiles, ...) resolve.
repo = Path("/content/nairobi-risk-mapping")
if repo.exists():
    os.chdir(repo)
    sys.path.insert(0, str(repo))
    for name in ("data", "checkpoints", "outputs"):
        src = DRIVE_ROOT / name
        dst = repo / name
        src.mkdir(parents=True, exist_ok=True)
        if not dst.exists() and not dst.is_symlink():
            dst.symlink_to(src, target_is_directory=True)

GEE_PROJECT_ID = os.environ.get("GEE_PROJECT_ID", "mtl-nairobi-2026")
REBUILD_GRAPHS = False          # True if buildings GeoJSON is missing from an older Sprint 1/3/4 run
NRRP_GEOJSON_PATH = None        # e.g. str(DRIVE_ROOT / "nrrp_high_risk_zones.geojson")
RIVER_BUFFER_M = 120
UNET_CHECKPOINT = Path("checkpoints/attention_unet_best.pt")
GRAPHSAGE_CHECKPOINT = Path("checkpoints/graphsage_best.pt")
print("DRIVE_ROOT =", DRIVE_ROOT.resolve())
print("cwd =", Path.cwd())

## 1. Repo check — Sprint 5 APIs exist


In [ ]:
from src import config
from src.models.fusion_engine import (
    FusionEngine,
    align_node_outputs_to_parcels,
    dump_policy_report,
    fuse_settlements,
    policy_nrrp_report,
    summarise_policy,
)
from src.models.graphsage_model import GraphSAGEModel
from src.models.attention_unet_model import AttentionUNetModel
from src.models.dataset import DEFAULT_LAYOUT, EVENT_S1_LAYOUT
from src.models.multi_settlement import load_settlement_graphs
from src.utils.parcel_io import (
    load_buildings_geojson,
    load_nrrp_or_river_reference,
    predict_flood_surface,
    sample_array_at_xy,
    sample_raster_at_points,
    union_geometries,
)
from src.data_pipeline import DataPipeline

print("policy classes:", config.POLICY_CLASS_LABELS)
print("flood bins: low <", config.POLICY_FLOOD_LOW_MAX, "; high >=", config.POLICY_FLOOD_HIGH_MIN)
print("default matrix:\n", config.DEFAULT_POLICY_MATRIX)
print("NRRP lift target:", config.POLICY_NRRP_LIFT_TARGET)
assert hasattr(FusionEngine, "classify")
assert callable(policy_nrrp_report) and callable(fuse_settlements)
print("Sprint 5 APIs OK")

## 2. Load graphs + building footprints

Sprint 5 needs the PyG graph (GraphSAGE node order) **and** `{settlement}_buildings.geojson` (footprints). Older `acquire_graph_data` runs only wrote the `.pt` file — set `REBUILD_GRAPHS = True` once to persist the GeoJSON beside it.


In [ ]:
from src import config as cfg

SETTLEMENT_KEYS = list(cfg.SETTLEMENT_KEYS)
graphs_dir = Path("data/graphs")
graphs_dir.mkdir(parents=True, exist_ok=True)

need_rebuild = REBUILD_GRAPHS or not all(
    (graphs_dir / f"{k}_graph.pt").exists()
    and (graphs_dir / f"{k}_buildings.geojson").exists()
    for k in SETTLEMENT_KEYS
)
if need_rebuild:
    for key in SETTLEMENT_KEYS:
        print("=" * 60, key)
        pipe = DataPipeline(settlement_key=key, gee_project_id=GEE_PROJECT_ID)
        pipe.acquire_graph_data()
else:
    print("Using existing graphs + buildings GeoJSON in", graphs_dir.resolve())

graphs = load_settlement_graphs(graphs_dir, settlement_keys=SETTLEMENT_KEYS)
parcels = {}
for key in SETTLEMENT_KEYS:
    path = graphs_dir / f"{key}_buildings.geojson"
    if path.exists():
        parcels[key] = load_buildings_geojson(path)
        print(f"{key:8s}  graph nodes={graphs[key].num_nodes if key in graphs else '—'}  "
              f"parcels={len(parcels[key])}")
    else:
        print(f"{key:8s}  missing buildings GeoJSON")

## 3. GraphSAGE deprivation probabilities

Load the Sprint 4 checkpoint when present and run `predict_proba`. If the checkpoint is missing this cell leaves `dep_proba` empty and later sections fall back to the OSM-proxy **labels** on `data.y` — useful to exercise the fusion API, not a substitute for a trained head in the report.


In [ ]:
import numpy as np

dep_proba = {}
gs_model = None
if graphs:
    sample = next(iter(graphs.values()))
    gs_model = GraphSAGEModel(in_channels=sample.x.size(1))
    if GRAPHSAGE_CHECKPOINT.exists():
        gs_model.load_checkpoint(str(GRAPHSAGE_CHECKPOINT))
        print("Loaded GraphSAGE checkpoint", GRAPHSAGE_CHECKPOINT)
        gs_source = "checkpoint"
    else:
        print("No GraphSAGE checkpoint at", GRAPHSAGE_CHECKPOINT,
              "— fusing OSM-proxy labels (one-hot of data.y), not a trained model.")
        gs_source = "label_onehot"

    for key, data in graphs.items():
        if gs_source == "checkpoint":
            proba = gs_model.predict_proba(data).numpy()
        else:
            y = data.y.cpu().numpy()
            proba = np.eye(cfg.DEPRIVATION_NUM_CLASSES)[y]
        if key in parcels:
            node_ids = getattr(data, "node_ids", None)
            if node_ids is None:
                node_ids = np.arange(proba.shape[0])
            else:
                node_ids = node_ids.cpu().numpy()
            dep_proba[key] = align_node_outputs_to_parcels(
                proba, node_ids, parcels[key]["node_id"].to_numpy(),
            )
            print(f"{key:8s}  deprivation proba {dep_proba[key].shape}  "
                  f"argmax counts={np.bincount(dep_proba[key].argmax(1), minlength=3).tolist()}")
else:
    gs_source = "missing"
    print("No graphs loaded.")

## 4. Flood probability at each parcel

Preferred path: run the trained Attention U-Net over the settlement GeoTIFF with overlapping 256 px windows, mean-blend the overlap, then sample the surface at building centroids.

Fallback: sample the `flood_label` band of the same stack. That is the **event-change label**, not a model output — the notebook prints which path ran. Do not quote label-sampled numbers as U-Net flood probabilities.


In [ ]:
import rasterio
from rasterio.transform import xy as raster_xy

flood_p = {}
flood_source = {}
layout = EVENT_S1_LAYOUT  # Sprint 2 v2 / Sprint 4 default; switch to DEFAULT_LAYOUT for 10-band stacks

def _find_stack(key):
    drive = DRIVE_ROOT
    candidates = [
        drive / f"{key}_raster_stack.tif",
        drive / f"{key}_raster_stack_event.tif",
        Path("data/raster_stacks") / f"{key}_raster_stack.tif",
    ]
    for p in candidates:
        if p.exists():
            return p
    return None


def _predict_tile_fn(model):
    def predict_tile(chip):
        tensor = torch.from_numpy(np.asarray(chip)).float().unsqueeze(0)
        return model.predict_proba(tensor)[0, 0].numpy()
    return predict_tile


unet_model = None
if UNET_CHECKPOINT.exists():
    # Infer in_channels from the checkpoint / layout.
    unet_model = AttentionUNetModel(layout=layout, dropout=0.2)
    try:
        unet_model.load_checkpoint(str(UNET_CHECKPOINT))
        print("Loaded U-Net checkpoint", UNET_CHECKPOINT)
    except Exception as exc:
        print("Could not load U-Net checkpoint:", exc)
        unet_model = None

for key, gdf in parcels.items():
    stack_path = _find_stack(key)
    if stack_path is None:
        print(f"{key:8s}  no raster stack — skip flood sampling")
        continue
    with rasterio.open(stack_path) as src:
        stack = src.read()
        transform = src.transform
        raster_crs = src.crs
        band_names = [n for n in src.descriptions if n]
    if not band_names:
        band_names = list(layout.band_names)[: stack.shape[0]]

    used_layout = layout if stack.shape[0] == len(layout.band_names) else DEFAULT_LAYOUT
    sar = {i for i, n in enumerate(band_names) if n == "vv" or str(n).endswith("_vv")}
    normed = DataPipeline._normalise_bands(stack.astype(np.float32), sar_band_indices=sar)

    if unet_model is not None and normed.shape[0] >= max(used_layout.input_indices) + 1:
        surface = predict_flood_surface(
            normed, used_layout.input_indices, _predict_tile_fn(unet_model),
        )
        flood_source[key] = "unet"
    else:
        # Label-band stand-in. Honest: this is the training target, not the head.
        surface = np.nan_to_num(stack[used_layout.label_index], nan=0.0)
        flood_source[key] = "flood_label_band"

    gdf_r = gdf.to_crs(raster_crs) if gdf.crs is not None else gdf
    centroids = gdf_r.geometry.centroid
    sampled = sample_array_at_xy(
        surface, transform, centroids.x.to_numpy(), centroids.y.to_numpy(),
    )
    sampled = np.nan_to_num(sampled, nan=0.0)
    flood_p[key] = sampled
    print(f"{key:8s}  flood source={flood_source[key]:16s}  "
          f"p_mean={sampled.mean():.3f}  p_p90={np.quantile(sampled, 0.9):.3f}")

## 5. Fuse heads and write the parcel policy layer

`FusionEngine` concatenates `[p_flood, p_low, p_med, p_high]` and indexes the 3×3 rule matrix in `config.DEFAULT_POLICY_MATRIX`. Changing those cuts does **not** require retraining either head.


In [ ]:
engine = FusionEngine()
ready = [k for k in SETTLEMENT_KEYS if k in parcels and k in flood_p and k in dep_proba]
print("Settlements ready to fuse:", ready)

if not ready:
    print("Nothing to fuse — need parcels + flood probabilities + deprivation probabilities.")
    fused = None
else:
    fused = fuse_settlements(
        parcels_by_settlement={k: parcels[k] for k in ready},
        flood_by_settlement={k: flood_p[k] for k in ready},
        deprivation_by_settlement={k: dep_proba[k] for k in ready},
        engine=engine,
        output_dir=Path(cfg.CLASSIFICATIONS_DIR),
        reference_gdf=None,  # NRRP scored in the next section
    )
    print("Wrote", fused["combined_path"])
    print("Pooled counts:", fused["summary"]["counts"])
    for key, row in fused["summary"]["per_settlement"].items():
        print(f"  {key:8s} {row['counts']}")

## 6. Validate Compensated Resettlement against NRRP (or river corridor)

Sprint 5's quantitative gate is **resettlement lift**: how many times more concentrated the relocate class is inside the independent high-risk reference than a spatially random assignment would be. Lift ≥ `config.POLICY_NRRP_LIFT_TARGET` (1.5) is the bar. Retain-and-Formalise lift should sit *below* 1 — we should not be handing tenure security to plots inside the overlay.

Reference priority: NRRP polygons if `NRRP_GEOJSON_PATH` is set, else an OSM river/stream buffer. Real accuracy still needs the 37-zone file in Colab; the river fallback is a coarser riparian prior, not a substitute for NRRP.


In [ ]:
import geopandas as gpd

nrrp_reports = {}
if not ready:
    print("Skip NRRP validation — fusion did not run.")
else:
    # One AOI covering all ready settlements so a single NRRP file is reused.
    aoi = gpd.GeoDataFrame(
        geometry=[union_geometries(parcels[k].to_crs(cfg.CRS_WGS84)) for k in ready],
        crs=cfg.CRS_WGS84,
    )
    aoi = gpd.GeoDataFrame(geometry=[union_geometries(aoi)], crs=cfg.CRS_WGS84)

    reference_gdf, ref_source = load_nrrp_or_river_reference(
        aoi, nrrp_path=NRRP_GEOJSON_PATH, river_buffer_m=RIVER_BUFFER_M,
    )
    print("Reference:", ref_source)

    if reference_gdf is None:
        print("No NRRP / river reference — supply NRRP_GEOJSON_PATH for the Sprint 5 lift statistic.")
    else:
        fused = fuse_settlements(
            parcels_by_settlement={k: parcels[k] for k in ready},
            flood_by_settlement={k: flood_p[k] for k in ready},
            deprivation_by_settlement={k: dep_proba[k] for k in ready},
            engine=engine,
            output_dir=Path(cfg.CLASSIFICATIONS_DIR),
            reference_gdf=reference_gdf,
        )
        report = fused["nrrp_report"]
        print("=" * 60)
        print("SPRINT 5 NRRP / RIPARIAN VALIDATION")
        print("=" * 60)
        print(f"reference          : {ref_source}")
        print(f"parcels            : {report['n_parcels']}")
        print(f"inside reference   : {report['n_inside_reference']} "
              f"({100*report['reference_fraction']:.1f}%)")
        print(f"resettlement lift  : {report['resettlement_lift']:.2f}  "
              f"(target >= {report['lift_target']})  meets={report['meets_lift_target']}")
        print(f"formalise lift     : {report['formalise_lift']:.2f}  "
              f"depleted_in_reference={report['formalise_depleted_in_reference']}")
        for name, row in report["per_class"].items():
            print(f"  {row['label']:28s}  n={row['support']:5d}  "
                  f"inside={row['n_inside_reference']:5d}  lift={row['lift']:.2f}")
        dump_policy_report(fused, Path(cfg.CLASSIFICATIONS_DIR) / "sprint5_policy_report.json")
        print("Wrote", Path(cfg.CLASSIFICATIONS_DIR) / "sprint5_policy_report.json")

## 7. Threshold summary


In [ ]:
print("=" * 60)
print("SPRINT 5 THRESHOLD SUMMARY")
print("=" * 60)
print("Policy matrix (rows=flood bin low/med/high, cols=deprivation low/med/high):")
print(np.asarray(engine.policy_matrix))
print("Flood cuts: low <", engine.flood_low_max, "; high >=", engine.flood_high_min)
if fused is None:
    print("Fusion did not run — no pooled counts.")
else:
    print("Pooled policy counts:", fused["summary"]["counts"])
    nrrp = fused.get("nrrp_report")
    if nrrp is None:
        print("NRRP lift: not scored (no reference).")
    else:
        print("Resettlement lift:", round(nrrp["resettlement_lift"], 3),
              "meets_target=", nrrp["meets_lift_target"])
        print("Formalise depleted inside reference:", nrrp["formalise_depleted_in_reference"])
print()
print("Flood probability source per settlement:", flood_source)
print("Deprivation source:", gs_source)
print()
print("Do not treat label-sampled flood probabilities or one-hot OSM-proxy")
print("deprivation as reportable model outputs. Re-run with Sprint 4")
print("checkpoints + NRRP polygons in Colab for the numbers that go in the report.")
print("Next: Sprint 6 Grad-CAM / SHAP + Folium dashboard (do not start here).")

## Done

- `FusionEngine` is the standalone class the proposal's OOAD design asked for: flood probability + deprivation probability in, three-class policy out, matrix retunable without retraining.
- Classified GeoJSON lands in `outputs/classifications/` (per settlement + combined).
- NRRP (or river-corridor) lift is the Sprint 5 validation statistic; real 37-zone polygons still need a Colab run.
- Sprint 6 owns Grad-CAM, SHAP, and the Folium dashboard — they are stubbed on the model wrappers until then.
